# PIPELINE 3 

## Config 

In [1]:
"""
PH Address Parsing Pipeline  (v2)
==================================
Stage 0 — Load reference data
Stage 1 — Junk token stripping
Stage 2 — Alias normalization
Stage 3 — City detection  (right-to-left, exact → fuzzy, multi-candidate)
Stage 4 — Barangay fuzzy matching  (blocked by city)
Stage 5 — Confidence scoring
Stage 6 — Export  (valid / partial / invalid)
"""

import os
import re
import json
import unicodedata
from datetime import datetime
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz, process
from openpyxl.styles import Font, PatternFill, Alignment

In [2]:
DIM_LOC_PATH  = "../data/mapping/dim_location_20260421.csv"
ALIAS_PATH    = "../data/utils/ph_address_alias_extended_v6.csv"
INPUT_PATH    = "../data/input/batch7_subset_01.csv"
INPUT_COL     = "order_deliveraddress"


OUTPUT_DIR    = "../data/output/"
VALID_DIR     = os.path.join(OUTPUT_DIR, "valid")
PARTIAL_DIR   = os.path.join(OUTPUT_DIR, "partial")
INVALID_DIR   = os.path.join(OUTPUT_DIR, "invalid")

for d in [VALID_DIR, PARTIAL_DIR, INVALID_DIR]:
    os.makedirs(d, exist_ok=True)

# Fuzzy thresholds — tune these against your data
CITY_FUZZY_THRESHOLD   = 85   # token_set_ratio: city name match in address segment
BGY_FUZZY_THRESHOLD    = 70   # token_sort_ratio: barangay name match (higher = fewer false positives)
CONFIDENCE_VALID       = 75   # composite score >= this → valid
CONFIDENCE_PARTIAL     = 45   # composite score >= this → partial, else invalid


## Helper

In [3]:
def strip_accents(text: str) -> str:
    """Remove diacritical marks (ñ → n, é → e, etc.)."""
    return "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )

def clean_str(s: str) -> str:
    """Lowercase, strip accents, collapse whitespace."""
    s = strip_accents(str(s)).lower()
    return re.sub(r"\s+", " ", s).strip()


## Load Reference Data

In [4]:
# ── 2. LOAD REFERENCE DATA ────────────────────────────────────────────────────

print("\nLoading reference data …")

# dim_location  (42k+ barangay rows)
dim_raw = pd.read_csv(DIM_LOC_PATH, encoding="iso-8859-1") #encoding='utf-8', encoding='latin-1' iso-8859-1
dim_raw.columns = dim_raw.columns.str.strip()
for col in ["barangay_name", "city_name", "province_name", "region_name"]:
    dim_raw[col] = dim_raw[col].astype(str).str.strip()

dim_raw["bgy_clean"]  = dim_raw["barangay_name"].apply(clean_str)
dim_raw["city_clean"] = dim_raw["city_name"].apply(clean_str)
dim_raw["prov_clean"] = dim_raw["province_name"].apply(clean_str)

# ── OPT 1: Pre-sort cities once — never sort inside the per-address loop ──────
all_cities_clean  = {clean_str(c): c for c in dim_raw["city_name"].dropna().unique()}
_sorted_cities    = sorted(all_cities_clean.items(), key=lambda x: -len(x[0]))
all_city_cleans   = [c for c, _ in _sorted_cities]   # for rapidfuzz list

# ── OPT 2: Compile one mega alternation regex — replaces 1407-iteration loop ──
# Sorted longest-first so the regex engine matches the most specific name first.
# e.g. "San Fernando" matches before "San" would.
_city_alts   = "|".join(re.escape(city_c) for city_c, _ in _sorted_cities)
CITY_MEGA_RE = re.compile(r"\b(?:" + _city_alts + r")\b", re.IGNORECASE)
print(f"  City mega-regex compiled ({len(_sorted_cities)} cities) ✓")

# ── OPT 3a: Precompute city subsets dict — O(1) lookup replaces df filter ─────
# city_clean → DataFrame subset for that city (used in barangay matching)
_city_subsets: dict[str, pd.DataFrame] = {
    city_c: grp.copy()
    for city_c, grp in dim_raw.groupby("city_clean")
}
print(f"  City subsets precomputed ({len(_city_subsets)} keys) ✓")

# ── OPT 3b: Precompute barangay exact-match dict — O(1) replaces df filter ───
# bgy_clean → [(city_name, province_name), ...]
_bgy_exact_dict: dict[str, list[tuple[str, str]]] = {}
for _, row in dim_raw.iterrows():
    _bgy_exact_dict.setdefault(row["bgy_clean"], []).append(
        (row["city_name"], row["province_name"])
    )
print(f"  Barangay exact dict precomputed ({len(_bgy_exact_dict):,} keys) ✓")

# ── OPT 3c: Precompute stripped barangay list for fuzzy fallback ──────────────
# Stripping "barangay" prefix avoids false 100-score matches on "barangay X"
_BARANGAY_PREFIX_RE = re.compile(r"^\s*barangay\s*", re.IGNORECASE)
_bgy_stripped_entries: list[tuple[str, str, str, str]] = [
    (row["bgy_clean"],
     _BARANGAY_PREFIX_RE.sub("", row["bgy_clean"]).strip(),
     row["city_name"],
     row["province_name"])
    for _, row in dim_raw.iterrows()
]
_bgy_stripped_list = [x[1] for x in _bgy_stripped_entries]
print(f"  Barangay stripped list precomputed ({len(_bgy_stripped_list):,} entries) ✓")

# Alias map  (abbreviation/shorthand → canonical form)
alias_df = pd.read_csv(ALIAS_PATH, encoding="iso-8859-1", usecols=["pattern", "replacement"])
alias_df = alias_df.dropna(subset=["pattern", "replacement"])
alias_df["pattern"]     = alias_df["pattern"].astype(str).str.strip()
alias_df["replacement"] = alias_df["replacement"].astype(str).str.strip()
alias_df = alias_df.sort_values("pattern", key=lambda s: s.str.len(), ascending=False)
alias_map = dict(zip(alias_df["pattern"].str.upper(), alias_df["replacement"].str.upper()))

print(f"  dim_location rows : {len(dim_raw):,}")
print(f"  alias rules       : {len(alias_map):,}")
print(f"  unique cities     : {len(all_cities_clean):,}")
print("Reference data loaded ✓")




Loading reference data …
  City mega-regex compiled (1405 cities) ✓
  City subsets precomputed (1406 keys) ✓
  Barangay exact dict precomputed (25,756 keys) ✓
  Barangay stripped list precomputed (42,011 entries) ✓
  dim_location rows : 42,011
  alias rules       : 535
  unique cities     : 1,405
Reference data loaded ✓


## Load Address

In [5]:
print("\nLoading addresses …")
input_df      = pd.read_csv(INPUT_PATH)
RAW_ADDRESSES = input_df[INPUT_COL].dropna().tolist()
print(f"  {len(RAW_ADDRESSES):,} addresses loaded")


Loading addresses …
  200,000 addresses loaded


## MAIN LINE

### Junk Token Stripping 

In [6]:
# ── 4. STAGE 1 — JUNK TOKEN STRIPPING ────────────────────────────────────────
#
# Removes noise that hurts fuzzy matching:
#   • Stray punctuation (backticks, tildes)
#   • Phone numbers
#   • Landmark phrases  (near, beside, in front of, across, opp.)
#   • Lot / Block / Unit / Floor / Room designations
#   • Building / subdivision / compound names — unified single pass that strips
#     0-4 PREFIX words + the keyword + 0-4 SUFFIX words in one go.
#     This prevents the prefix being left orphaned when the keyword is removed first.
#   • Street-level numbers (3-5 digits) — kept: 2-digit brgy numbers
#
# Meaningful abbreviations are protected before stripping and restored after.
# e.g.  A.C → ANGELES_CITY → restored after junk removal

_BUILDING_KW = (
    r"bldg|building|tower|plaza|centre|center|subd|subdivision|compound|cmpd|"
    r"village|vill|estate|residences|residencia|condominium|condo|"
    r"apartelle|apartment|apt|annex|mall|square|complex|commercial|industrial|zone|cluster"
)

_STREET_TYPES = (
    r"street|avenue|boulevard|road|highway|lane|drive|circle|"
    r"place|extension|alley|parkway|terrace|walk|"
    r"st|ave|blvd|rd|hwy|dr|ln|pkwy|ter|extn|ext"
)

_PROTECT_PATTERNS = [
    # ── Core city abbreviations ──────────────────────────────────────────────
    (re.compile(r"\bA\.C\.?\b", re.IGNORECASE), "ANGELES_CITY"),
    (re.compile(r"\bQ\.C\.?\b", re.IGNORECASE), "QUEZON_CITY"),
    (re.compile(r"\bM\.C\.?\b", re.IGNORECASE), "MAKATI_CITY"),  # sometimes appears in datasets
    (re.compile(r"\bB\.G\.C\.?\b", re.IGNORECASE), "TAGUIG"),
    (re.compile(r"\bBGC\b", re.IGNORECASE), "TAGUIG"),

    # ── Common shorthand / aliases ───────────────────────────────────────────
    (re.compile(r"\bQC\b", re.IGNORECASE), "QUEZON_CITY"),
    (re.compile(r"\bMNL\b", re.IGNORECASE), "MANILA"),
    (re.compile(r"\bMM\b", re.IGNORECASE), "METRO_MANILA"),
    (re.compile(r"\bNCR\b", re.IGNORECASE), "METRO_MANILA"),

    # ── “City” compressed / malformed tokens ─────────────────────────────────
    (re.compile(r"\bQCITY\b", re.IGNORECASE), "QUEZON_CITY"),
    (re.compile(r"\bQZNCITY\b", re.IGNORECASE), "QUEZON_CITY"),
    (re.compile(r"\bMKTCITY\b", re.IGNORECASE), "MAKATI_CITY"),
    (re.compile(r"\bMLACITY\b", re.IGNORECASE), "MANILA"),

    # ── No-space variants (VERY common in raw data) ───────────────────────────
    (re.compile(r"\bQUEZONCITY\b", re.IGNORECASE), "QUEZON_CITY"),
    (re.compile(r"\bMAKATICITY\b", re.IGNORECASE), "MAKATI_CITY"),
    (re.compile(r"\bANGELESCITY\b", re.IGNORECASE), "ANGELES_CITY"),
    (re.compile(r"\bDAVAO(CITY)?\b", re.IGNORECASE), "DAVAO_CITY"),

    # ── Barangay shorthand (helps downstream barangay detection too) ─────────
    (re.compile(r"\bBRGY\.?\b", re.IGNORECASE), "BARANGAY"),
    (re.compile(r"\bBRG\.?\b", re.IGNORECASE), "BARANGAY"),
    (re.compile(r"\bBGY\.?\b", re.IGNORECASE), "BARANGAY"),

    # ── Street compression protection (prevents over-stripping) ──────────────
    (re.compile(r"\bST\.?\b", re.IGNORECASE), "STREET"),
    (re.compile(r"\bRD\.?\b", re.IGNORECASE), "ROAD"),
    (re.compile(r"\bAVE\.?\b", re.IGNORECASE), "AVENUE"),

    # ── Province abbreviations (useful for your +15 scoring bonus) ────────────
    (re.compile(r"\bPAMP\b", re.IGNORECASE), "PAMPANGA"),
    (re.compile(r"\bBUL\b", re.IGNORECASE), "BULACAN"),
    (re.compile(r"\bLAG\b", re.IGNORECASE), "LAGUNA"),

]

_STRAY_PUNCT      = re.compile(r"[`~|]")
_PHONE_NUMBERS    = re.compile(r"\b(0|\+63)\d{9,10}\b")
_STREET_NUMBERS   = re.compile(r"(?<!\w)\d{3,5}(?!\w)")
_LOT_BLK_UNIT     = re.compile(
    r"\b(lot|blk|block|unit|floor|flr|fl|rm|room|door|phase|house no|hse no|#)"
    r"\s*[.\-]?\s*[\w\-]*",
    re.IGNORECASE,
)
_LANDMARK_PHRASES = re.compile(
    r"\b(near|beside|in front of|across from|across|opposite|opp\.?|behind|"
    r"adjacent to|adj\.?|along|corner of|corner|cor\.?)\b[^,]*",
    re.IGNORECASE,
)

# Unified building pattern: [0-4 prefix words] [keyword] [0-4 suffix words]
# Suffix stops at known address anchors (Brgy, St, Ave, Road) to prevent over-stripping.
_BUILDING_UNIFIED = re.compile(
    r"\b(?:[A-Za-z]\w*\.?\s+){0,4}"
    r"\b(?:" + _BUILDING_KW + r")\b\.?"
    r"(?:\s+(?!(?:brgy|barangay|st\b|ave\b|road\b|blvd\b))[A-Za-z]\w*){0,4}",
    re.IGNORECASE,
)

# Street phrase pattern: [0-4 name words] [street type] [optional compass suffix]
# Strips the ENTIRE phrase — not just the type word — to prevent false city/bgy matches.
# e.g. "Santa Ana Street" → "" (not "Santa Ana" which would false-match Santa Ana city)
# e.g. "Rizal Ave, Angeles City" → ", Angeles City" (comma segment boundary preserved)
_STREET_PHRASE = re.compile(
    r"\b(?:[A-Za-z]\w*\.?\s+){0,4}"
    r"\b(?:" + _STREET_TYPES + r")\b\.?"
    r"(?:\s+(?:north|south|east|west|extension|ext))?",
    re.IGNORECASE,
)


def strip_junk(addr: str) -> str:
    """
    Remove noise tokens from a raw address string.
    Returns a cleaner string ready for alias normalization.
    Extend _BUILDING_KW, _STREET_TYPES, or _PROTECT_PATTERNS as new patterns emerge.
    """
    s = addr
    # Protect meaningful abbreviations
    for pattern, placeholder in _PROTECT_PATTERNS:
        s = pattern.sub(placeholder, s)

    s = _STRAY_PUNCT.sub(" ", s)
    s = _PHONE_NUMBERS.sub(" ", s)
    s = _LANDMARK_PHRASES.sub(" ", s)
    s = _LOT_BLK_UNIT.sub(" ", s)
    s = _BUILDING_UNIFIED.sub(" ", s)   # single unified pass — prefix+keyword+suffix
    s = _STREET_PHRASE.sub(" ", s)      # strip whole street phrase, not just type word
    s = _STREET_NUMBERS.sub(" ", s)

    # Restore protected tokens
    s = s.replace("ANGELES_CITY", "Angeles City")

    s = re.sub(r"\s{2,}", " ", s)
    s = re.sub(r"(,\s*){2,}", ", ", s)
    s = re.sub(r"^[\s,\.]+|[\s,\.]+$", "", s)
    return s.strip()


print("Junk stripper defined ✓")

Junk stripper defined ✓


### Alias Normalization 

In [7]:
# ── 5. STAGE 2 — ALIAS NORMALIZATION ─────────────────────────────────────────
#
# Applies alias_map token-by-token (up to 3-word windows, longest-first).
# Carried over from the current pipeline — this part works well.

def normalize_address(addr: str) -> str:
    """
    Apply alias replacements on up to 3-word windows, longest-first.
    Returns title-cased normalized string.
    """
    upper  = addr.upper()
    tokens = re.split(r"(\s+|,|\.)", upper)
    result = []
    i = 0
    while i < len(tokens):
        tok = tokens[i].strip()
        if not tok or tok in (",", "."):
            result.append(tokens[i])
            i += 1
            continue
        matched = False
        for lookahead in (3, 2, 1):
            candidate_tokens, j, count = [], i, 0
            while j < len(tokens) and count < lookahead:
                if re.match(r"^\s*$", tokens[j]) or tokens[j] in (",", "."):
                    j += 1
                    continue
                candidate_tokens.append(tokens[j])
                j += 1
                count += 1
            candidate = " ".join(candidate_tokens)
            if candidate in alias_map:
                result.append(alias_map[candidate])
                i = j
                matched = True
                break
        if not matched:
            result.append(tokens[i])
            i += 1
    normalized = re.sub(r"\s+", " ", "".join(result)).strip()
    # Remove stray dots left when "BRGY." is tokenized as BRGY + "."
    normalized = re.sub(r"\.\s+", " ", normalized)
    normalized = re.sub(r"\s{2,}", " ", normalized).strip()
    return normalized.title()


print("Alias normalizer defined ✓")


# ── 5b. STAGE 2.5 — USELESS ADDRESS FILTER ───────────────────────────────────
#
# Applied AFTER strip_junk() + normalize_address(), BEFORE city detection.
# Catches addresses that have no matchable location content and short-circuits
# them directly to the "invalid" bucket — avoiding wasted fuzzy scan time.
#
# An address is useless if ANY of these are true:
#   A — Null/placeholder value  (N/A, None, null, xxx, -, .)
#   B — No alphabetic content at all  (e.g. "12-A", "123", ".")
#   C — Zero meaningful tokens after removing stop words + numbers
#       (e.g. "Barangay", "Brgy 22", "Street", "Avenue")
#   D — Only region-level tokens with no city/bgy specificity
#       (e.g. "Metro Manila", "NCR", "Luzon")
#
# "Meaningful token" = not a stop word, not a pure number, len >= 2.
# Single meaningful token = borderline — kept unless it's a region-only word,
# because it could be a standalone barangay or city name (handled by bgy fallback).

# Build once from your dim_location table
KNOWN_LOCATION_TOKENS = set()

KNOWN_LOCATION_TOKENS.update(dim_raw["city_name"].str.lower())
KNOWN_LOCATION_TOKENS.update(dim_raw["province_name"].str.lower())
KNOWN_LOCATION_TOKENS.update(dim_raw["barangay_name"].str.lower())


_STRONG_LOCATION_PATTERNS = re.compile(
    r"\b(city|city of)\b", re.IGNORECASE
)

_NULL_PATTERNS = re.compile(
    r"^(n/?a|none|null|xxx+|\-+|\.+|0+|tba|tbd|na|nd|unknown|N\.A\.?)$",
    re.IGNORECASE,
)

_REGION_ONLY_TOKENS = {
    "metro manila", "ncr", "luzon", "visayas", "mindanao",
    "national capital region", "calabarzon", "bicol", "ilocos",
    "cagayan valley", "central luzon", "mimaropa",
}

_ADDRESS_STOP_WORDS = {
    "barangay", "brgy", "bgy", "bry", "brg", "brngy",
    "street", "avenue", "road", "boulevard", "highway", "lane",
    "drive", "place", "extension", "corner", "near", "beside",
    "the", "a", "an", "of", "and", "or",
    "st", "ave", "rd", "dr", "blvd", "hwy",
    "na", "none", "null", ",", "-", "/", "(", ")", "#",
    "house", "unit", "room", "floor", "lot", "block", "blk", "phase",
    "north", "south", "east", "west",
}


def _meaningful_tokens(addr_c: str) -> list[str]:
    tokens = re.split(r"[\s,\.\-/\(\)#]+", addr_c.lower())

    return [
        t for t in tokens
        if t
        and (
            t in KNOWN_LOCATION_TOKENS   # 🔥 preserve real locations
            or (
                t not in _ADDRESS_STOP_WORDS
                and not re.match(r"^[\d][\d\-a-z]*$", t)
                and len(t) >= 2
            )
        )
    ]

def is_useless(norm_addr: str) -> tuple[bool, str]:
    """
    Returns (True, reason_code) when the address has no matchable location content.
    Returns (False, "") when the address should proceed to city detection.

    Call this after normalize_address() and before detect_city_candidates().
    """
    addr_c = clean_str(norm_addr)

    # A — null / placeholder
    if not addr_c or _NULL_PATTERNS.match(addr_c.strip()):
        return True, "null_placeholder"

    # B — no alphabetic content
    if not re.search(r"[a-zA-Z]{2,}", addr_c):
        return True, "no_alphabetic_content"

    # C — zero meaningful tokens
    mt = _meaningful_tokens(addr_c)
    if len(mt) == 0:
        return True, "zero_meaningful_tokens"

    # D — region-only content
    if len(mt) <= 2:
        if all(t in _REGION_ONLY_TOKENS for t in mt):
            return True, "region_only"
        
    # 🔥 STRONG SIGNAL BYPASS
    if _STRONG_LOCATION_PATTERNS.search(addr_c):
        return False, ""
    
        # If any known city/province appears → NOT useless
    if any(tok in KNOWN_LOCATION_TOKENS for tok in addr_c.split()):
        return False, ""
    
    return False, ""


print("Useless filter defined ✓")

Alias normalizer defined ✓
Useless filter defined ✓


### City Detection 

In [8]:

# ── 6. STAGE 3 — CITY DETECTION ──────────────────────────────────────────────
#
# detect_city_candidates() — unified for BOTH comma-delimited and space-delimited input.
#
# Phase A — Comma-segment scan (weighted RTL, existing behaviour)
#   Segments are read right-to-left. Weight: last=1.25, second=1.10, earlier=0.90.
#   Handles: "Brgy Cutcut, Angeles, Pampanga"
#
# Phase B — Token-stream scan (new, for space-delimited input)
#   Scans the full cleaned string with CITY_MEGA_RE.finditer() to find ALL city
#   name matches. Each match is scored by:
#     • Position weight  (end of string = 1.25, start = 0.85)
#       — PH addresses run specific → general left-to-right
#     • +15 bonus  if the city's province also appears in the address
#     • +10 bonus  if a known barangay of that city appears in remaining tokens
#   Handles: "Brgy Pampang Angeles City Pampanga"
#
# Province-as-city penalty (−35):
#   If city_c is also a province name AND another candidate has city_c as its
#   own province, city_c is a province modifier, not a city anchor.
#   Prevents "Bulacan" outscoring "San Jose Del Monte" in "San Jose Del Monte Bulacan".
#
# detect_via_barangay() — fallback when both phases yield nothing.
#   Unchanged: O(1) exact n-gram dict lookup → fuzzy fallback.

_STREET_TOKENS_RE = re.compile(
    r"\b(street|avenue|road|boulevard|highway|lane|drive|circle|place|extension)\b",
    re.IGNORECASE,
)

# Precompute: city_clean → set of province_clean (for province bonus + penalty)
_city_to_provs: dict[str, set[str]] = {}
for _, _row in dim_raw[["city_clean", "prov_clean"]].drop_duplicates().iterrows():
    _city_to_provs.setdefault(_row["city_clean"], set()).add(_row["prov_clean"])

# Precompute: set of all province_clean values (for province-as-city penalty)
_all_provs_clean: set[str] = set(dim_raw["prov_clean"].unique())


def _token_position_weight(match_start: int, total_len: int) -> float:
    """
    Tokens near the END of the address get higher weight.
    PH address order: specific (front) → general/city/province (back).
    Returns 0.85 at position 0 → 1.25 at final position.
    """
    pos = match_start / max(total_len - 1, 1)
    return 0.85 + 0.40 * pos

def _prep_for_bgy_match(norm_addr: str) -> str:
    """Strip barangay prefix, number tokens, and street types before bgy fuzzy match."""
    s = _BARANGAY_PREFIX_RE.sub("", clean_str(norm_addr)).strip()
    s = re.sub(r"\b[\d][\d\-a-z]*\b", "", s)   # e.g. 22, 12-A, 54
    s = _STREET_TOKENS_RE.sub("", s)
    return re.sub(r"\s{2,}", " ", s).strip()

def detect_city_candidates(norm_addr: str) -> list[tuple[str, str]]:
    """
    Returns [(city_clean, city_original), ...] sorted by confidence score.
    Works for comma-delimited, space-delimited, and mixed input.
    Returns at most 3 candidates (the barangay matcher disambiguates).
    """
    addr_c   = clean_str(norm_addr)
    addr_len = len(addr_c)
    candidates: dict[str, float] = {}   # city_clean → best score

    # ── Phase A: comma-segment scan ───────────────────────────────────────────
    segments = [s.strip() for s in norm_addr.split(",") if s.strip()]
    if len(segments) > 1:
        for seg_idx, seg in enumerate(reversed(segments)):
            seg_c = clean_str(seg)
            if len(seg_c) < 3:
                continue
            seg_w = 1.25 if seg_idx == 0 else (1.10 if seg_idx == 1 else 0.90)

            m = CITY_MEGA_RE.search(seg_c)
            if m:
                city_c = m.group(0).lower()
                candidates[city_c] = max(candidates.get(city_c, 0.0), 100 * seg_w)
                continue

            seg_nc = re.sub(r"\s{2,}", " ", re.sub(r"\bcity\b", "", seg_c)).strip()
            if seg_nc != seg_c:
                m2 = CITY_MEGA_RE.search(seg_nc)
                if m2:
                    city_c = m2.group(0).lower()
                    candidates[city_c] = max(candidates.get(city_c, 0.0), 100 * seg_w)
                    continue

            best = process.extractOne(seg_c, all_city_cleans, scorer=fuzz.token_set_ratio)
            if best and best[1] >= CITY_FUZZY_THRESHOLD:
                candidates[best[0]] = max(candidates.get(best[0], 0.0), best[1] * seg_w)

    # ── Phase B: token-stream scan ────────────────────────────────────────────
    for m in CITY_MEGA_RE.finditer(addr_c):
        city_c    = m.group(0).lower()
        pos_w     = _token_position_weight(m.start(), addr_len)
        score     = 100.0 * pos_w
        remaining = re.sub(r"\s{2,}", " ",
                           (addr_c[:m.start()] + addr_c[m.end():]).strip())

        # +15 bonus: province of this city is also present in the address
        for prov_c in _city_to_provs.get(city_c, set()):
            if prov_c and re.search(r"\b" + re.escape(prov_c) + r"\b", addr_c):
                score += 15
                break

        # +10 bonus: a barangay belonging to this city found in remaining tokens
        sub = _city_subsets.get(city_c)
        if sub is not None and remaining:
            bgy_q = _BARANGAY_PREFIX_RE.sub("", remaining).strip()
            bgy_q = re.sub(r"\b[\d][\d\-a-z]*\b", "", bgy_q).strip()
            if bgy_q:
                bgy_best = process.extractOne(
                    bgy_q, sub["bgy_clean"].tolist(),
                    scorer=fuzz.partial_ratio,
                    score_cutoff=BGY_FUZZY_THRESHOLD,
                )
                if bgy_best:
                    score += 10

        candidates[city_c] = max(candidates.get(city_c, 0.0), score)

    # City-suffix strip pass — handles alias-expanded "Angeles City" vs dim "Angeles"
    addr_nc = re.sub(r"\s{2,}", " ", re.sub(r"\bcity\b", "", addr_c)).strip()
    if addr_nc != addr_c:
        for m in CITY_MEGA_RE.finditer(addr_nc):
            city_c = m.group(0).lower()
            if city_c not in candidates:   # don't overwrite already-scored cities
                pos_w = _token_position_weight(m.start(), len(addr_nc))
                candidates[city_c] = 95.0 * pos_w   # slight discount vs direct match

    if not candidates:
        return []

    # ── Province-as-city penalty (−35) ────────────────────────────────────────
    # A city that is also a province name AND is the province of another candidate
    # is acting as a province modifier (e.g. "Bulacan" in "San Jose Del Monte Bulacan").
    # Penalise it so the more specific city name wins.
    province_roles: set[str] = set()
    for city_c in list(candidates):
        if city_c in _all_provs_clean:
            for other_c in candidates:
                if other_c != city_c and city_c in _city_to_provs.get(other_c, set()):
                    province_roles.add(city_c)
                    break
    for city_c in province_roles:
        candidates[city_c] -= 35

    # Sort descending by score, return top 3
    sorted_cands = sorted(candidates.items(), key=lambda x: -x[1])
    return [
        (c, all_cities_clean[c])
        for c, _ in sorted_cands[:3]
        if c in all_cities_clean
    ]

def detect_via_barangay(norm_addr: str) -> list[tuple[str, str]]:
    """
    Fallback: infer city from barangay name when city detection yields nothing.
    Returns [(city_original, province_original), ...].

    Uses _bgy_exact_dict for O(1) exact n-gram lookup instead of
    repeated df[df["bgy_clean"] == phrase] filter calls.
    """
    q = _prep_for_bgy_match(norm_addr)
    if len(q) < 3:
        return []

    # Step 1 — O(1) exact n-gram lookup (3 → 2 → 1 word windows)
    tokens = q.split()
    for n in (3, 2, 1):
        for i in range(len(tokens) - n + 1):
            phrase = " ".join(tokens[i:i + n])
            if len(phrase) < 4:
                continue
            if phrase in _bgy_exact_dict:
                seen: dict[str, str] = {}
                for city, prov in _bgy_exact_dict[phrase]:
                    seen.setdefault(city, prov)
                return list(seen.items())

    # Step 2 — fuzzy fallback over stripped barangay list
    best = process.extractOne(
        q, _bgy_stripped_list,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=BGY_FUZZY_THRESHOLD,
    )
    if best:
        idx   = _bgy_stripped_list.index(best[0])
        entry = _bgy_stripped_entries[idx]
        return [(entry[2], entry[3])]

    return []

print("City detector + barangay fallback defined ✓")


City detector + barangay fallback defined ✓


### Baranggay Fuzzy Matching 

In [9]:
# ── 7. STAGE 4 — BARANGAY FUZZY MATCHING ─────────────────────────────────────
#
# For each (city_clean, city_original) candidate:
#   • Sub-filter dim_location to that city
#   • Fuzzy-match the cleaned address against bgy_clean  (partial_ratio)
#   • Collect best (score, row) per candidate
# Then pick the overall best-scoring candidate.
#
# Returns a result dict that includes the match, its score, and a match_tier.

def _score_match(addr_c: str, city_c: str) -> tuple[float, pd.Series | None]:
    """
    Fuzzy-match addr_c against barangay names for city_c.
    Uses precomputed _city_subsets dict — O(1) lookup, no DataFrame filter.
    Returns (best_score, best_row).  score = 0.0 if no match found.
    """
    sub = _city_subsets.get(city_c)
    if sub is None or sub.empty:
        return 0.0, None
    bgy_names   = sub["bgy_clean"].tolist()
    best        = process.extractOne(addr_c, bgy_names, scorer=fuzz.partial_ratio)
    if best is None:
        return 0.0, None
    matched_row = sub[sub["bgy_clean"] == best[0]].iloc[0]
    return float(best[1]), matched_row


def match_address(row: pd.Series) -> dict:
    """
    Full matching for one address row.
    Returns a flat dict of all output columns.
    """
    norm           = row["normalized_address"]
    addr_c         = clean_str(norm)
    candidates     = list(row["city_candidates"])
    bgy_city_cands = row["bgy_city_candidates"]

    # ── Short-circuit: useless addresses flagged before city detection ─────────
    useless_reason = row.get("useless_reason", "")
    if useless_reason:
        return _build_result(
            orig_addr  = row["order_deliveraddress"],
            norm_addr  = norm,
            addr_c     = addr_c,
            matched_row= None,
            bgy_score  = 0.0,
            city_score = 0.0,
            tier       = "invalid",
            reason     = f"useless_{useless_reason}",
        )

    # ── Promote barangay fallback into candidates when city detect found nothing
    if not candidates and bgy_city_cands:
        candidates = [(clean_str(city), city) for city, prov in bgy_city_cands]

    # ── Nothing at all ────────────────────────────────────────────────────────
    if not candidates:
        return _build_result(
            orig_addr  = row["order_deliveraddress"],
            norm_addr  = norm,
            addr_c     = addr_c,
            matched_row= None,
            bgy_score  = 0.0,
            city_score = 0.0,
            tier       = "invalid",
            reason     = "no_city_detected",
        )

    # ── Try each city candidate, keep best barangay score ────────────────────
    best_bgy_score  = -1.0
    best_row        = None
    best_city_clean = None
    best_city_orig  = None

    for city_c, city_orig in candidates:
        bgy_score, matched_row = _score_match(addr_c, city_c)   # OPT: dict lookup, no df filter
        if bgy_score > best_bgy_score:
            best_bgy_score  = bgy_score
            best_row        = matched_row
            best_city_clean = city_c
            best_city_orig  = city_orig

    # City-presence score: is the detected city string literally in the address?
    city_score = fuzz.partial_ratio(best_city_clean, addr_c) if best_city_clean else 0.0

    # Composite confidence  (70% barangay match, 30% city presence)
    composite = 0.7 * best_bgy_score + 0.3 * city_score

    # ── Determine tier ────────────────────────────────────────────────────────
    if best_bgy_score >= BGY_FUZZY_THRESHOLD and composite >= CONFIDENCE_VALID:
        tier   = "valid"
        reason = "barangay_matched"
    elif composite >= CONFIDENCE_PARTIAL:
        tier   = "partial"
        reason = "city_only_or_low_confidence"
    else:
        tier   = "invalid"
        reason = "below_confidence_threshold"

    return _build_result(
        orig_addr   = row["order_deliveraddress"],
        norm_addr   = norm,
        addr_c      = addr_c,
        matched_row = best_row,
        bgy_score   = round(best_bgy_score, 1),
        city_score  = round(city_score, 1),
        tier        = tier,
        reason      = reason,
        city_orig   = best_city_orig,
        composite   = round(composite, 1),
    )


def _build_result(
    orig_addr, norm_addr, addr_c,
    matched_row, bgy_score, city_score,
    tier, reason,
    city_orig=None, composite=None,
) -> dict:
    """Assemble the output dict from a matched (or unmatched) row."""
    if matched_row is not None:
        return {
            "order_deliveraddress" : orig_addr,
            "normalized_address"   : norm_addr,
            "barangay_code"        : matched_row.get("barangay_code"),
            "barangay_name"        : matched_row["barangay_name"],
            "city_code"            : matched_row.get("city_code"),
            "city_name"            : matched_row["city_name"],
            "province_code"        : matched_row.get("province_code"),
            "province_name"        : matched_row["province_name"],
            "region_code"          : matched_row.get("region_code"),
            "region_name"          : matched_row["region_name"],
            "addr_clean"           : addr_c,
            "bgy_match_score"      : bgy_score,
            "city_match_score"     : city_score,
            "confidence_score"     : composite,
            "match_tier"           : tier,
            "match_reason"         : reason,
        }
    else:
        return {
            "order_deliveraddress" : orig_addr,
            "normalized_address"   : norm_addr,
            "barangay_code"        : None,
            "barangay_name"        : None,
            "city_code"            : None,
            "city_name"            : city_orig,
            "province_code"        : None,
            "province_name"        : None,
            "region_code"          : None,
            "region_name"          : None,
            "addr_clean"           : addr_c,
            "bgy_match_score"      : bgy_score,
            "city_match_score"     : city_score,
            "confidence_score"     : composite if composite is not None else 0.0,
            "match_tier"           : tier,
            "match_reason"         : reason,
        }


print("Matcher defined ✓")

Matcher defined ✓


### Running Main

In [10]:
# ── 8. RUN PIPELINE ───────────────────────────────────────────────────────────
#
# Single-threaded loop with tqdm progress bar.
# For 120k rows (~8 min single-thread), parallelise with:
#
#   from multiprocessing import Pool
#   with Pool(processes=os.cpu_count()) as pool:
#       results = pool.map(process_one, RAW_ADDRESSES)
#
# where process_one() wraps stages 1-5 for a single address.
# See the parallel runner template at the bottom of this file.

try:
    from tqdm import tqdm
    _iter = tqdm(RAW_ADDRESSES, desc="Parsing", unit="addr")
except ImportError:
    _iter = RAW_ADDRESSES
    print("  (install tqdm for a progress bar: pip install tqdm)")

print("\nRunning pipeline …")

records = []
for addr in _iter:
    # Stage 1 — strip junk (buildings, street phrases, landmarks, lot/unit tokens)
    stripped = strip_junk(addr)
    # Stage 2 — normalize aliases (BRGY→Barangay, A.C→Angeles City, etc.)
    normalized = normalize_address(stripped)
    # Stage 2.5 — useless filter (short-circuit before expensive fuzzy scans)
    useless, useless_reason = is_useless(normalized)
    if useless:
        records.append({
            "order_deliveraddress" : addr,
            "stripped_address"     : stripped,
            "normalized_address"   : normalized,
            "city_candidates"      : [],
            "bgy_city_candidates"  : [],
            "useless_reason"       : useless_reason,
        })
        continue
    # Stage 3a — detect city from address directly
    city_candidates = detect_city_candidates(normalized)
    # Stage 3b — barangay fallback (only computed when city detect fails)
    bgy_city_candidates = detect_via_barangay(normalized) if not city_candidates else []
    records.append({
        "order_deliveraddress" : addr,
        "stripped_address"     : stripped,
        "normalized_address"   : normalized,
        "city_candidates"      : city_candidates,
        "bgy_city_candidates"  : bgy_city_candidates,
        "useless_reason"       : "",
    })

working_df = pd.DataFrame(records)

# Stage 4+5 — match + score
matched_records = working_df.apply(match_address, axis=1).tolist()
results_df      = pd.DataFrame(matched_records)

print(f"Pipeline complete — {len(results_df):,} addresses processed")




Parsing:   0%|          | 0/200000 [00:00<?, ?addr/s]


Running pipeline …


Parsing:   1%|▏         | 2583/200000 [00:10<12:58, 253.55addr/s]


KeyboardInterrupt: 

### Segregate and Export 

In [ ]:
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment


def write_excel(df: pd.DataFrame, path: str, sheet: str = "Results"):
    """
    Write a styled Excel file with:
    - Frozen header row
    - Auto column width
    - Styled header (bold, white text, blue fill)
    """

    # Create Excel file
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name=sheet)
        ws = writer.sheets[sheet]

        # Auto-adjust column widths
        for col_cells in ws.columns:
            max_len = 0
            col_letter = col_cells[0].column_letter

            for cell in col_cells:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))

            ws.column_dimensions[col_letter].width = min(max_len + 4, 60)

        # Header styling
        header_fill = PatternFill(
            fill_type="solid",
            fgColor="1F4E79"
        )

        for cell in ws[1]:
            cell.font = Font(
                bold=True,
                color="FFFFFF",
                name="Arial",
                size=10
            )
            cell.fill = header_fill
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True
            )

        # Set header height + freeze pane
        ws.row_dimensions[1].height = 30
        ws.freeze_panes = "A2"

In [ ]:
# ── 9. STAGE 6 — SINGLE FILE EXPORT ──────────────────────────────────────────

import os
from datetime import datetime

OUTPUT_COLS = [
    "order_deliveraddress", "normalized_address",
    "barangay_code", "barangay_name",
    "city_code", "city_name",
    "province_code", "province_name",
    "region_code", "region_name",
    "bgy_match_score", "city_match_score", "confidence_score",
    "match_tier", "match_reason",
]


def ensure_cols(df, cols):
    """Ensure all expected columns exist and enforce column order."""
    for c in cols:
        if c not in df.columns:
            df[c] = None
    return df[cols].copy()


# 📊 Summary (monitoring only)
summary = results_df["match_tier"].value_counts()

print("\n📊 Match Summary:")
for k, v in summary.items():
    print(f"  {k.upper():<8}: {v:,}")


# 📁 Build timestamped output path (DATE + TIME)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(OUTPUT_DIR, f"address_parsed_{timestamp}.xlsx")


# 📌 Sort by confidence (best matches first)
results_sorted = results_df.sort_values(
    by="confidence_score",
    ascending=False
)


# 📦 Final export
final_df = ensure_cols(results_sorted, OUTPUT_COLS)

write_excel(
    final_df,
    output_path,
    sheet="Results"
)


print("\n✅ Export complete")
print(f"  File → {output_path}")


📊 Match Summary:
  VALID   : 138,090
  PARTIAL : 46,153
  INVALID : 15,757



✅ Export complete
  File → ../data/output/address_parsed_20260511_164129.xlsx


In [ ]:
# # ── 9. STAGE 6 — SEGREGATE & EXPORT ──────────────────────────────────────────

# OUTPUT_COLS = [
#     "order_deliveraddress", "normalized_address",
#     "barangay_code", "barangay_name",
#     "city_code",     "city_name",
#     "province_code", "province_name",
#     "region_code",   "region_name",
#     "bgy_match_score", "city_match_score", "confidence_score",
#     "match_tier", "match_reason",
# ]

# def ensure_cols(df, cols):
#     for c in cols:
#         if c not in df.columns:
#             df[c] = None
#     return df[cols].copy()

# valid_df   = results_df[results_df["match_tier"] == "valid"]
# partial_df = results_df[results_df["match_tier"] == "partial"]
# invalid_df = results_df[results_df["match_tier"] == "invalid"]

# # Break down invalid by reason for visibility
# useless_df = invalid_df[invalid_df["match_reason"].str.startswith("useless_", na=False)]
# low_conf_df = invalid_df[~invalid_df["match_reason"].str.startswith("useless_", na=False)]

# print(f"\n  ✅ Valid          : {len(valid_df):,}")
# print(f"  🟡 Partial        : {len(partial_df):,}")
# print(f"  ❌ Invalid total  : {len(invalid_df):,}")
# print(f"     ├─ Useless     : {len(useless_df):,}  (no location content — junk/empty/null)")
# print(f"     └─ Low-conf    : {len(low_conf_df):,}  (had content but couldn't resolve)")
# if len(useless_df):
#     reason_counts = useless_df["match_reason"].value_counts()
#     for reason, count in reason_counts.items():
#         print(f"        {reason}: {count:,}")


# def write_excel(df: pd.DataFrame, path: str, sheet: str = "Results"):
#     """Write a styled Excel file with frozen header row."""
#     with pd.ExcelWriter(path, engine="openpyxl") as writer:
#         df.to_excel(writer, index=False, sheet_name=sheet)
#         ws = writer.sheets[sheet]
#         for col_cells in ws.columns:
#             max_len = max((len(str(c.value)) if c.value else 0) for c in col_cells)
#             ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 4, 60)
#         header_fill = PatternFill("solid", fgColor="1F4E79")
#         for cell in ws[1]:
#             cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
#             cell.fill      = header_fill
#             cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
#         ws.row_dimensions[1].height = 30
#         ws.freeze_panes = "A2"
#     print(f"  Exported → {path}")


# date_tag     = datetime.now().strftime("%Y%m%d")
# valid_path   = os.path.join(VALID_DIR,   f"valid_{date_tag}.xlsx")
# partial_path = os.path.join(PARTIAL_DIR, f"partial_{date_tag}.xlsx")
# invalid_path = os.path.join(INVALID_DIR, f"invalid_{date_tag}.xlsx")

# write_excel(ensure_cols(valid_df,   OUTPUT_COLS), valid_path,   "Valid")
# write_excel(ensure_cols(partial_df, OUTPUT_COLS), partial_path, "Partial")
# write_excel(ensure_cols(invalid_df, OUTPUT_COLS), invalid_path, "Invalid")

# print(f"  Valid   → {valid_path}")
# print(f"  Partial → {partial_path}")
# print(f"  Invalid → {invalid_path}")


# # ══════════════════════════════════════════════════════════════════════════════
# # OPTIONAL: PARALLEL RUNNER  (~4x faster on 4 cores, ~8x on 8 cores)
# # ══════════════════════════════════════════════════════════════════════════════
# #
# # Replace the run loop above with this block for large batches (50k+ rows).
# # All precomputed structures (CITY_MEGA_RE, _city_subsets, _bgy_exact_dict,
# # _bgy_stripped_list) are module-level globals and are inherited by worker
# # processes via fork (Linux/Mac) without re-serialisation overhead.
# #
# # Usage: uncomment, set N_WORKERS, replace the run loop section above.
# #
# # import os
# # from multiprocessing import Pool
# #
# # N_WORKERS = os.cpu_count()   # or set explicitly, e.g. 8
# #
# # def _process_one(addr: str) -> dict:
# #     """Stages 1-5 for a single address — safe to call from worker processes."""
# #     stripped            = strip_junk(addr)
# #     normalized          = normalize_address(stripped)
# #     city_candidates     = detect_city_candidates(normalized)
# #     bgy_city_candidates = detect_via_barangay(normalized) if not city_candidates else []
# #     row = pd.Series({
# #         "order_deliveraddress" : addr,
# #         "stripped_address"     : stripped,
# #         "normalized_address"   : normalized,
# #         "city_candidates"      : city_candidates,
# #         "bgy_city_candidates"  : bgy_city_candidates,
# #     })
# #     return match_address(row)
# #
# # if __name__ == "__main__":   # required guard for multiprocessing on Windows
# #     with Pool(processes=N_WORKERS) as pool:
# #         matched_records = list(tqdm(
# #             pool.imap(_process_one, RAW_ADDRESSES, chunksize=500),
# #             total=len(RAW_ADDRESSES), desc="Parsing (parallel)", unit="addr"
# #         ))
# #     results_df = pd.DataFrame(matched_records)



# Validation

In [ ]:
""" validate_parsed_addresses.py
─────────────────────────────
Post-processing validator for the PH Address Parsing Pipeline output.

For each row it checks whether the extracted barangay, city, and province
actually appear in the original raw address string (or its normalised form).
Values that cannot be verified are nullified to prevent hallucinations.
A match_accuracy_score (0–100) is added to every row.

Usage
─────
    from validate_parsed_addresses import validate_parsed_addresses

    results_df = run_pipeline(...)          # your existing pipeline call
    validated  = validate_parsed_addresses(results_df)

The function is non-destructive: it works on a copy and never modifies the
DataFrame you pass in.
"""

import re
import unicodedata
from typing import Optional

import pandas as pd
from rapidfuzz import fuzz


# ── Thresholds ────────────────────────────────────────────────────────────────
# Barangay: fuzzy ≥ 75 — typos/plurals are common in raw addresses
# City    : fuzzy ≥ 85 — city was the detection anchor, should be accurate
# Province: exact only — province is always inferred from city, rarely stated
#           literally in raw; absence is expected and is NOT penalised

_BGY_FUZZY_FLOOR  = 75
_CITY_FUZZY_FLOOR = 85
_PROV_FUZZY_FLOOR = None    # None = no fuzzy fallback; exact match only

# Score weights (sum = 6)
_W_BARANGAY = 3   # most specific field, highest hallucination risk
_W_CITY     = 2   # primary detection anchor
_W_PROVINCE = 1   # inferred; absence is not penalised
_MAX_SCORE  = _W_BARANGAY + _W_CITY + _W_PROVINCE   # = 6

# Required input columns — pipeline must have produced these
_REQUIRED_COLS = {
    "order_deliveraddress",
    "normalized_address",
    "barangay_name",
    "city_name",
    "province_name",
    "barangay_code",
    "city_code",
    "match_tier",
    "bgy_match_score",
}


# ── Internal helpers ──────────────────────────────────────────────────────────

def _strip_accents(text: str) -> str:
    """Remove diacritical marks (ñ → n, é → e, etc.)."""
    return "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )


def _normalise(s: str) -> str:
    """Lowercase, strip accents, collapse whitespace."""
    return re.sub(r"\s+", " ", _strip_accents(str(s)).lower()).strip()


def _value_in_text(
    value: str,
    text: str,
    fuzzy_floor: Optional[int],
) -> tuple[bool, float]:
    """
    Check whether `value` appears in `text`.

    Step 1 — Exact word-boundary match.
              Fastest path; handles the majority of clean matches.
    Step 2 — Fuzzy partial_ratio fallback (only when fuzzy_floor is not None).
              Handles plural/typo variants (e.g. "Additions Hills" → "Addition Hills").

    Returns
    ───────
    (found: bool, score: float 0–100)
        score = 100.0 on exact match, partial_ratio score on fuzzy match,
        0.0 when not found.
    """
    if not value or not text:
        return False, 0.0

    v = _normalise(value)
    t = _normalise(text)

    # Step 1: exact word-boundary
    if re.search(r"\b" + re.escape(v) + r"\b", t):
        return True, 100.0

    # Step 2: fuzzy fallback
    if fuzzy_floor is None:
        return False, float(fuzz.partial_ratio(v, t))

    score = fuzz.partial_ratio(v, t)
    return (score >= fuzzy_floor), float(score)


def _check_field(
    value,
    raw_addr: str,
    norm_addr: str,
    fuzzy_floor: Optional[int],
) -> tuple[bool, float]:
    """
    Check a single extracted field against BOTH the raw and the normalised address.
    Credited if found in either — alias expansion (A.C → Angeles) may mean
    the extracted value only appears in the normalised form.

    Returns (found: bool, best_score: float)
    """
    if not value or (isinstance(value, float) and pd.isna(value)):
        return False, 0.0

    found_raw,  score_raw  = _value_in_text(str(value), raw_addr,  fuzzy_floor)
    found_norm, score_norm = _value_in_text(str(value), norm_addr, fuzzy_floor)

    found = found_raw or found_norm
    score = max(score_raw, score_norm)
    return found, score


def _validate_row(row: pd.Series) -> pd.Series:
    """
    Validate one result row.

    Nullification rules
    ───────────────────
    barangay_name / barangay_code:
        Nullified when the barangay cannot be found in either the raw or
        the normalised address. Exception: if the pipeline already confirmed
        the match with a high bgy_match_score (≥ 70) AND match_tier == "valid",
        we trust the pipeline — the barangay may have been inferred from
        context and not be literally stated in the raw string.

    city_name / city_code:
        Nullified only when the city cannot be found AND the pipeline's own
        city_match_score is also low (< 60). The city was the primary
        detection anchor and is rarely wrong — we apply a stricter bar.

    province_name:
        Never nullified. Province is always inferred from city and is
        almost never stated literally in PH raw addresses. Its absence
        in the raw string is expected and is not penalised.
        If city_name is nullified, province_name is also cleared (they
        are linked — a province without a city is meaningless).

    Scoring
    ───────
    match_accuracy_score = round((weighted_pts / 6) * 100, 1)
        barangay found → +3 pts
        city found     → +2 pts
        province found → +1 pt
    """
    raw_addr  = str(row.get("order_deliveraddress", ""))
    norm_addr = str(row.get("normalized_address",   ""))
    match_tier     = str(row.get("match_tier",      ""))
    bgy_match_score = float(row.get("bgy_match_score", 0.0) or 0.0)
    city_match_score = float(row.get("city_match_score", 0.0) or 0.0)

    bgy_name  = row.get("barangay_name")
    city_name = row.get("city_name")
    prov_name = row.get("province_name")

    notes: list[str] = []

    # ── Barangay validation ───────────────────────────────────────────────────
    bgy_found, bgy_val_score = _check_field(
        bgy_name, raw_addr, norm_addr, _BGY_FUZZY_FLOOR
    )

    pipeline_confirmed_bgy = (
        match_tier == "valid" and bgy_match_score >= 70
    )

    if bgy_found or pipeline_confirmed_bgy:
        bgy_pts = _W_BARANGAY
        val_bgy_found = True
        if pipeline_confirmed_bgy and not bgy_found:
            notes.append("bgy:pipeline_confirmed_not_in_raw")
    else:
        # Nullify — cannot verify, likely hallucination
        bgy_pts = 0
        val_bgy_found = False
        bgy_name = None
        row = row.copy()
        row["barangay_name"] = None
        row["barangay_code"] = None
        notes.append("bgy:nullified_not_found")

    # ── City validation ───────────────────────────────────────────────────────
    city_found, city_val_score = _check_field(
        city_name, raw_addr, norm_addr, _CITY_FUZZY_FLOOR
    )

    # Only nullify city if BOTH the text search AND the pipeline's own score are low
    pipeline_city_weak = city_match_score < 60

    if city_found or not pipeline_city_weak:
        city_pts = _W_CITY
        val_city_found = True
        if not city_found and not pipeline_city_weak:
            notes.append("city:pipeline_score_ok_not_in_raw")
    else:
        city_pts = 0
        val_city_found = False
        row = row.copy()
        row["city_name"] = None
        row["city_code"] = None
        # Cascade: without a city, province is meaningless
        row["province_name"] = None
        row["province_code"] = None
        notes.append("city:nullified_not_found")

    # ── Province validation ───────────────────────────────────────────────────
    # Province is inferred — check for scoring only, never nullified standalone
    effective_prov = row.get("province_name")   # may have been cleared above
    if effective_prov and not (isinstance(effective_prov, float) and pd.isna(effective_prov)):
        prov_found, prov_val_score = _check_field(
            effective_prov, raw_addr, norm_addr, _PROV_FUZZY_FLOOR
        )
    else:
        prov_found, prov_val_score = False, 0.0

    prov_pts = _W_PROVINCE if prov_found else 0
    val_prov_found = prov_found

    # ── Compute final score ───────────────────────────────────────────────────
    raw_pts            = bgy_pts + city_pts + prov_pts
    match_accuracy_score = round((raw_pts / _MAX_SCORE) * 100, 1)

    # ── Build output series ───────────────────────────────────────────────────
    result = row.copy()
    result["val_barangay_found"]   = val_bgy_found
    result["val_city_found"]       = val_city_found
    result["val_province_found"]   = val_prov_found
    result["val_bgy_score"]        = round(bgy_val_score, 1)
    result["val_city_score"]       = round(city_val_score, 1)
    result["val_province_score"]   = round(prov_val_score, 1)
    result["match_accuracy_score"] = match_accuracy_score
    result["val_notes"]            = " | ".join(notes) if notes else "ok"

    return result


# ── Public API ────────────────────────────────────────────────────────────────

def validate_parsed_addresses(df: pd.DataFrame) -> pd.DataFrame:
    """
    Validate address parsing results and return a corrected DataFrame.

    For each row:
      1. Checks if barangay_name, city_name, province_name appear in
         order_deliveraddress (raw) and/or normalized_address (expanded).
      2. Nullifies unverifiable barangay / city values to prevent hallucinations.
      3. Adds match_accuracy_score (0–100) based on weighted field presence.

    Scoring weights
    ───────────────
      barangay_name  → 3 pts  (most specific, highest risk)
      city_name      → 2 pts  (primary anchor)
      province_name  → 1 pt   (inferred, absence expected)
      ─────────────────────────────────────────
      max total      → 6 pts  → normalised to 100

    Added columns
    ─────────────
      val_barangay_found    bool   — barangay verified in raw / normalised
      val_city_found        bool   — city verified in raw / normalised
      val_province_found    bool   — province verified in raw / normalised
      val_bgy_score         float  — best match score for barangay (0–100)
      val_city_score        float  — best match score for city (0–100)
      val_province_score    float  — best match score for province (0–100)
      match_accuracy_score  float  — weighted accuracy score (0–100)
      val_notes             str    — pipe-separated audit notes

    Corrected columns (may be set to None)
    ───────────────────────────────────────
      barangay_name, barangay_code  — nullified when unverifiable
      city_name, city_code          — nullified when unverifiable
      province_name, province_code  — cleared when city is nullified

    Parameters
    ──────────
    df : pd.DataFrame
        Output DataFrame from the PH address parsing pipeline.
        Must contain: order_deliveraddress, normalized_address,
        barangay_name, city_name, province_name, barangay_code,
        city_code, match_tier, bgy_match_score.

    Returns
    ───────
    pd.DataFrame
        A new DataFrame (copy) with corrected columns and validation metadata.

    Raises
    ──────
    ValueError
        If required pipeline columns are missing.
    """
    missing = _REQUIRED_COLS - set(df.columns)
    if missing:
        raise ValueError(
            f"Input DataFrame is missing required columns: {sorted(missing)}\n"
            f"Run the address parsing pipeline first."
        )

    if df.empty:
        return df.copy()

    validated = df.apply(_validate_row, axis=1)
    return validated.reset_index(drop=True)


# ── Summary helper ────────────────────────────────────────────────────────────

def validation_summary(validated_df: pd.DataFrame) -> None:
    """
    Print a concise summary of validation results.
    Call after validate_parsed_addresses().
    """
    n = len(validated_df)
    if n == 0:
        print("Empty DataFrame — nothing to summarise.")
        return

    bgy_found  = validated_df["val_barangay_found"].sum()
    city_found = validated_df["val_city_found"].sum()
    prov_found = validated_df["val_province_found"].sum()

    bgy_nulled  = validated_df["barangay_name"].isna().sum()
    city_nulled = validated_df["city_name"].isna().sum()

    avg_score = validated_df["match_accuracy_score"].mean()
    high_conf = (validated_df["match_accuracy_score"] >= 83.4).sum()   # all 3 fields found
    mid_conf  = (validated_df["match_accuracy_score"].between(50, 83.3)).sum()
    low_conf  = (validated_df["match_accuracy_score"] < 50).sum()

    note_counts = (
        validated_df["val_notes"]
        .str.split(" | ")
        .explode()
        .value_counts()
    )

    print("═" * 55)
    print("  ADDRESS VALIDATION SUMMARY")
    print("═" * 55)
    print(f"  Total rows           : {n:,}")
    print(f"  Avg accuracy score   : {avg_score:.1f} / 100")
    print()
    print(f"  FIELD VERIFICATION")
    print(f"  ├─ Barangay found    : {bgy_found:,} / {n:,}  ({bgy_found/n*100:.1f}%)")
    print(f"  ├─ City found        : {city_found:,} / {n:,}  ({city_found/n*100:.1f}%)")
    print(f"  └─ Province found    : {prov_found:,} / {n:,}  ({prov_found/n*100:.1f}%)")
    print()
    print(f"  NULLIFICATIONS")
    print(f"  ├─ Barangay nulled   : {bgy_nulled:,}  (hallucination prevention)")
    print(f"  └─ City nulled       : {city_nulled:,}  (unverifiable anchor)")
    print()
    print(f"  SCORE DISTRIBUTION")
    print(f"  ├─ High  ≥83.4 (2-3 fields) : {high_conf:,}  ({high_conf/n*100:.1f}%)")
    print(f"  ├─ Mid   50–83 (1-2 fields) : {mid_conf:,}  ({mid_conf/n*100:.1f}%)")
    print(f"  └─ Low   < 50  (0-1 fields) : {low_conf:,}  ({low_conf/n*100:.1f}%)")
    if not note_counts.empty:
        print()
        print("  AUDIT NOTES (top issues)")
        for note, cnt in note_counts.head(6).items():
            if note != "ok":
                print(f"  ├─ {note:<38}: {cnt:,}")
    print("═" * 55)

## Run Validation

In [ ]:
# from validate_parsed_addresses import validate_parsed_addresses

# validated  = validate_parsed_addresses(pd.read_excel("VALIDATE.xlsx"))
validated  = validate_parsed_addresses(results_df)



In [ ]:
validated.to_excel(os.path.join(OUTPUT_DIR, f"validated_addresses_{timestamp}.xlsx"), index=False)